In [1]:
!pip install torchtext torchdata --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [2]:
!pip install spacy

In [3]:
!python3 -m spacy download en_core_web_sm


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "/usr/local/lib/python3.11/dist-packages/spacy/__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "/usr/local/lib/python3.11/dist-packages/spacy/errors.py", line 3, in <module>
    from .compat import Literal
  File "/usr/local/lib/python3.11/dist-packages/spacy/compat.py", line 4, in <module>
    from thinc.

In [4]:
!pip install "numpy<2"

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires torchdata==0.11.0, but you have torchdata 0.7.1 which is incompatible.
torchvision 0.21.0+cu124 requires torch==2.6.0, but you have torch 2.2.0+cu118 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [5]:
!pip install portalocker>=2.0.0

In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [36]:
import torch, torchdata, torchtext
from torch import nn

import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

#make our work comparable if restarted the kernel
SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

cpu


In [37]:
from torchtext.datasets import AG_NEWS
train, test = AG_NEWS()

In [38]:
train_size = len(list(iter(train)))
too_much, train, valid = train.random_split(total_length=train_size, weights = {"too_much": 0.7, "smaller_train": 0.2, "valid": 0.1}, seed=999)

In [39]:
from torchtext.data.utils import get_tokenizer
tokenizer = get_tokenizer('spacy', language='en_core_web_sm')
tokens = tokenizer("We are learning torchtext in AIT!")  #some test
tokens

['We', 'are', 'learning', 'torchtext', 'in', 'AIT', '!']

In [40]:
from torchtext.vocab import build_vocab_from_iterator
def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(yield_tokens(train), specials=['<unk>', '<pad>', '<bos>', '<eos>'])
vocab.set_default_index(vocab["<unk>"])

In [41]:
text_pipeline  = lambda x: vocab(tokenizer(x))
label_pipeline = lambda x: int(x) - 1 #turn {1, 2, 3, 4} to {0, 1, 2, 3} for pytorch training

In [42]:
from torch.utils.data   import DataLoader
from torch.nn.utils.rnn import pad_sequence

pad_idx = vocab["<pad>"] #++<----making sure our embedding layer ignores pad

def collate_batch(batch):
    label_list, text_list, length_list = [], [], []
    for (_label, _text) in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        length_list.append(processed_text.size(0))  #++<-----packed padded sequences require
    # criterion expects float labels
    return torch.tensor(label_list, dtype=torch.int64), pad_sequence(text_list,  padding_value=pad_idx, batch_first=True), torch.tensor(length_list, dtype=torch.int64)

In [43]:
batch_size = 64

train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(valid, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_loader  = DataLoader(test, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

In [44]:
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, output_dim, num_layers, bidirectional, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hid_dim * 2, output_dim)

    def forward(self, text, text_lengths):
        embedded = self.embedding(text)
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, text_lengths.to('cpu'), enforce_sorted=False, batch_first=True)
        packed_output, (hn, cn) = self.lstm(packed_embedded)  # if no h0, all zeroes
        output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        hn = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim = 1)
        return self.fc(hn)

In [45]:
from torchtext.vocab import FastText
fast_vectors = FastText(language='simple') #small for easy training
fast_embedding = fast_vectors.get_vecs_by_tokens(vocab.get_itos(), lower_case_backup=True).to(device)

In [46]:
fast_embedding.shape

torch.Size([52828, 300])

In [47]:
input_dim  = len(vocab)
hid_dim    = 256
emb_dim    = fast_embedding.size(1)   # 300
output_dim = 4 #four classes

#for biLSTM
num_layers = 2
bidirectional = True
dropout = 0.5

model = LSTM(input_dim, emb_dim, hid_dim, output_dim, num_layers, bidirectional, dropout).to(device)
# model.embedding.weight.data = fast_embedding
model.embedding.weight.data.copy_(fast_embedding)  #**<------applied the fast text embedding as the initial weights
# model.embedding.weight.requires_grad = False  # freeze if you dont want to train

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [-0.0307, -0.2038,  0.0423,  ..., -0.2226,  0.1934, -0.1614],
        [ 0.1253, -0.1663,  0.2053,  ...,  0.0221, -0.5225, -0.0705],
        [ 0.5765, -0.0857, -0.2223,  ..., -0.0556,  0.1672,  0.1642]])

In [48]:
criterion = nn.CrossEntropyLoss() #combine softmax with cross entropy

In [49]:
def accuracy(preds, y):
    predicted = torch.max(preds.data, 1)[1]
    batch_corr = (predicted == y).sum()
    acc = batch_corr / len(y)

    return acc

In [50]:
def evaluate(model, loader, criterion, loader_length):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()

    with torch.no_grad():
        for i, (label, text, text_length) in enumerate(loader):
            label = label.to(device) #(batch_size, )
            text  = text.to(device)  #(seq len, batch_size)

            predictions = model(text, text_length).squeeze(1)

            loss = criterion(predictions, label)
            acc  = accuracy(predictions, label)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / loader_length, epoch_acc / loader_length

In [51]:
test_loader_length  = len(list(iter(test_loader)))

In [52]:
save_path = f'/content/{model.__class__.__name__}.pt'
torch.save(model.state_dict(), save_path)
model.load_state_dict(torch.load(save_path, map_location=torch.device('cpu')))
test_loss, test_acc = evaluate(model, test_loader, criterion, test_loader_length)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Test Loss: 1.387 | Test Acc: 25.17%


### Pruning
Let's try prune and see the effect on the accuracy. Use `torch.nn.utils.prune` to prune our neural networks. We can also implement our own custom pruning technique. Traditionally, pruning is a technique for improving latency, reducing model size, and computational complexity as pruning makes most of the weights to zero.  Model pruning refers to the act of removing unimportant parameters from a deep learning neural network model to reduce the model size.

In [53]:
from torch.nn.utils import prune as prune

def reset():
    model = LSTM(input_dim, emb_dim, hid_dim, output_dim, num_layers, bidirectional, dropout).to(device)
    model.load_state_dict(torch.load(save_path, map_location=torch.device('cpu')))
    return model

In [54]:
fc = model.fc

In [55]:
fc

Linear(in_features=512, out_features=4, bias=True)

In [56]:
print(list(fc.named_parameters()))  # Let's see the weights. Notice weight and bias.

[('weight', Parameter containing:
tensor([[ 0.0426,  0.0199, -0.0126,  ..., -0.0379, -0.0259, -0.0220],
        [-0.0389, -0.0013, -0.0246,  ...,  0.0429, -0.0094,  0.0007],
        [-0.0148,  0.0003,  0.0002,  ..., -0.0161, -0.0162,  0.0190],
        [ 0.0106,  0.0279,  0.0244,  ...,  0.0332, -0.0118,  0.0288]],
       requires_grad=True)), ('bias', Parameter containing:
tensor([-0.0080,  0.0169,  0.0113, -0.0252], requires_grad=True))]


In [57]:
# for l in list(fc.named_parameters()):
#     print(f"{l[0]}, {l[1].detach().numpy().shape}")

for l in list(fc.named_parameters()):
    print(l[0], ':', l[1].detach().numpy().shape)

weight : (4, 512)
bias : (4,)


Let's also see the buffers. Buffer is simply a tensor but does not involved in gradient update. Buffer is like hidden variable and is useful to store different things (is like hidden in html)

In [58]:
print(list(fc.named_buffers()))

[]


To prune a module, there are many pruning techniques available in `torch.nn.utils.prune` (or implement your own by subclassing `BasePruningMethod`).

#### Random Pruning
We can prune at random 95% of the connections in the parameter named weight in the fc layer.

In [59]:
prune.random_unstructured(fc, name="weight", amount=0.95)

Linear(in_features=512, out_features=4, bias=True)

Notice the fc layers weight is now replaced with weight_orig, which is basically the original one.

In [60]:
print(list(fc.named_parameters()))

[('bias', Parameter containing:
tensor([-0.0080,  0.0169,  0.0113, -0.0252], requires_grad=True)), ('weight_orig', Parameter containing:
tensor([[ 0.0426,  0.0199, -0.0126,  ..., -0.0379, -0.0259, -0.0220],
        [-0.0389, -0.0013, -0.0246,  ...,  0.0429, -0.0094,  0.0007],
        [-0.0148,  0.0003,  0.0002,  ..., -0.0161, -0.0162,  0.0190],
        [ 0.0106,  0.0279,  0.0244,  ...,  0.0332, -0.0118,  0.0288]],
       requires_grad=True))]


How the pruning happens is, when forwarding, pytorch will apply the `weight_mask` stored in `named_buffers` to `weight_orig`. weight_mask has a bunch of 0 and 1, where this mask will be multipled with weight_orig resulting in some weight pruned.

In [61]:
print(list(fc.named_buffers()))

[('weight_mask', tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.]]))]


The pruning techniques implemented in torch.nn.utils.prune compute the pruned version of the weight (by combining the mask with the original parameter) and store them in the attribute `weight`. Note, this is no longer a parameter of the fc, it is now simply an attribute.

In [62]:
print(fc.weight)

tensor([[0.0000, 0.0000, -0.0000,  ..., -0.0000, -0.0000, -0.0000],
        [-0.0000, -0.0000, -0.0000,  ..., 0.0000, -0.0000, 0.0000],
        [-0.0000, 0.0000, 0.0000,  ..., -0.0000, -0.0000, 0.0000],
        [0.0000, 0.0000, 0.0244,  ..., 0.0000, -0.0000, 0.0000]],
       grad_fn=<MulBackward0>)


PyTorch prune the network really during forward function. Actually, pruning is applied prior to each forward pass using PyTorch's  `forward_pre_hooks`. Specifically, when the fc is pruned, it will acquire a `forward_pre_hook` for each parameter associated with it that gets pruned. In this case, since we have so far only pruned the original parameter named fc, only one hook will be present.

In [63]:
fc._forward_pre_hooks

OrderedDict([(1, <torch.nn.utils.prune.RandomUnstructured at 0x7f9ec0876310>)])

Let's try classification using our randomly pruned version.

In [64]:
test_loss, test_acc = evaluate(model, test_loader, criterion, test_loader_length)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Test Loss: 1.386 | Test Acc: 24.98%


Wow, pruning 95% still get 24.98% accuracy! In case you want to make your model permanently pruned, just use the command remove.

In [65]:
prune.remove(fc, "weight")

Linear(in_features=512, out_features=4, bias=True)

In [66]:
print(list(fc.named_parameters())) # notice now the weight is the pruned version and weight_orig is gone.

[('bias', Parameter containing:
tensor([-0.0080,  0.0169,  0.0113, -0.0252], requires_grad=True)), ('weight', Parameter containing:
tensor([[0.0000, 0.0000, -0.0000,  ..., -0.0000, -0.0000, -0.0000],
        [-0.0000, -0.0000, -0.0000,  ..., 0.0000, -0.0000, 0.0000],
        [-0.0000, 0.0000, 0.0000,  ..., -0.0000, -0.0000, 0.0000],
        [0.0000, 0.0000, 0.0244,  ..., 0.0000, -0.0000, 0.0000]],
       requires_grad=True))]


#### Magnitude Pruning
We can also prune less randomly, but based on the lowest magnitude, which indicates that those parameters has less importance.

In [72]:
model = reset()
fc = model.fc

In [73]:
print(list(fc.named_parameters()))

[('weight', Parameter containing:
tensor([[ 0.0426,  0.0199, -0.0126,  ..., -0.0379, -0.0259, -0.0220],
        [-0.0389, -0.0013, -0.0246,  ...,  0.0429, -0.0094,  0.0007],
        [-0.0148,  0.0003,  0.0002,  ..., -0.0161, -0.0162,  0.0190],
        [ 0.0106,  0.0279,  0.0244,  ...,  0.0332, -0.0118,  0.0288]],
       requires_grad=True)), ('bias', Parameter containing:
tensor([-0.0080,  0.0169,  0.0113, -0.0252], requires_grad=True))]


In [74]:
# prune based on the lowest magnitude based on l1 norm
prune.l1_unstructured(fc, name="weight", amount=0.95)

Linear(in_features=512, out_features=4, bias=True)

We can prune based on L2 norm on specific dim, using `ln_structured`, where structured means on particular dimension, while n refers to the type of norm.

In [ ]:
# prune based on the lowest magnitude based on l<n> norm
# prune.ln_structured(fc, name="weight", amount=0.95, n=2, dim=0)  # l2norm

In [75]:
print(list(fc.named_parameters()))

[('bias', Parameter containing:
tensor([-0.0080,  0.0169,  0.0113, -0.0252], requires_grad=True)), ('weight_orig', Parameter containing:
tensor([[ 0.0426,  0.0199, -0.0126,  ..., -0.0379, -0.0259, -0.0220],
        [-0.0389, -0.0013, -0.0246,  ...,  0.0429, -0.0094,  0.0007],
        [-0.0148,  0.0003,  0.0002,  ..., -0.0161, -0.0162,  0.0190],
        [ 0.0106,  0.0279,  0.0244,  ...,  0.0332, -0.0118,  0.0288]],
       requires_grad=True))]


In [76]:
print(list(fc.named_buffers()))

[('weight_mask', tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]))]


In [77]:
fc.weight

tensor([[0.0426, 0.0000, -0.0000,  ..., -0.0000, -0.0000, -0.0000],
        [-0.0000, -0.0000, -0.0000,  ..., 0.0429, -0.0000, 0.0000],
        [-0.0000, 0.0000, 0.0000,  ..., -0.0000, -0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, -0.0000, 0.0000]],
       grad_fn=<MulBackward0>)

In [78]:
fc._forward_pre_hooks

OrderedDict([(2, <torch.nn.utils.prune.L1Unstructured at 0x7f9ec0933110>)])

In [79]:
test_loss, test_acc = evaluate(model, test_loader, criterion, test_loader_length)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Test Loss: 1.386 | Test Acc: 25.13%


#### Pruning multiple parameters in a model
By specifying the desired pruning technique and parameters, we can easily prune multiple tensors in a network.

In [80]:
model = reset()

for name, module in model.named_modules():
    print(name)


embedding
lstm
fc


In [81]:
for name, module in model.named_modules():
     # prune 99% of connections in all embedding layers
    if isinstance(module, torch.nn.Embedding):
        prune.l1_unstructured(module, name="weight", amount=0.99) # name is nothing but parameter name in named_parameters()
    # prune 50% of connections in all linear layers
    elif isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=0.5)

print(dict(model.named_buffers()).keys())  # to verify that all masks exist

dict_keys(['embedding.weight_mask', 'fc.weight_mask'])


In [82]:
test_loss, test_acc = evaluate(model, test_loader, criterion, test_loader_length)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Test Loss: 1.387 | Test Acc: 25.05%


#### Global pruning

So far, we only looked at what is usually referred to as "local" pruning, i.e. the practice of pruning tensors in a model one by one, by comparing the statistics (weight magnitude, activation, gradient, etc.) of each entry exclusively to the other entries in that tensor.

However, a common and perhaps more powerful technique is to prune the model all at once, by removing (for example) the lowest 20% of connections across the whole model, instead of removing the lowest 20% of connections in each layer. This is likely to result in different pruning percentages per layer.

We can `global_unstructured` from torch.nn.utils.prune.

In [83]:
model = reset()
model

LSTM(
  (embedding): Embedding(52828, 300, padding_idx=1)
  (lstm): LSTM(300, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=512, out_features=4, bias=True)
)

In [84]:
for name, param in model.named_parameters():
    print(name)

embedding.weight
lstm.weight_ih_l0
lstm.weight_hh_l0
lstm.bias_ih_l0
lstm.bias_hh_l0
lstm.weight_ih_l0_reverse
lstm.weight_hh_l0_reverse
lstm.bias_ih_l0_reverse
lstm.bias_hh_l0_reverse
lstm.weight_ih_l1
lstm.weight_hh_l1
lstm.bias_ih_l1
lstm.bias_hh_l1
lstm.weight_ih_l1_reverse
lstm.weight_hh_l1_reverse
lstm.bias_ih_l1_reverse
lstm.bias_hh_l1_reverse
fc.weight
fc.bias


In [85]:
parameters_to_prune = (
    (model.embedding, 'weight'),
    (model.lstm, 'weight_ih_l0'),
    (model.lstm, 'weight_hh_l0'),
    (model.fc, 'weight'),
    (model.fc, 'bias')
)

prune.global_unstructured(parameters_to_prune, amount=0.7, pruning_method=prune.L1Unstructured)

In [86]:
test_loss, test_acc = evaluate(model, test_loader, criterion, test_loader_length)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Test Loss: 1.386 | Test Acc: 24.99%


In [87]:
print(
    "Sparsity in embedding.weight: {:.2f}%".format(
        100. * float(torch.sum(model.embedding.weight == 0))
        / float(model.embedding.weight.nelement())
    )
)
print(
    "Sparsity in lstm.weight_ih_l0: {:.2f}%".format(
        100. * float(torch.sum(model.lstm.weight_ih_l0 == 0))
        / float(model.lstm.weight_ih_l0.nelement())
    )
)
print(
    "Sparsity in lstm.weight_hh_l0: {:.2f}%".format(
        100. * float(torch.sum(model.lstm.weight_hh_l0 == 0))
        / float(model.lstm.weight_hh_l0.nelement())
    )
)
print(
    "Sparsity in fc.weight: {:.2f}%".format(
        100. * float(torch.sum(model.fc.weight == 0))
        / float(model.fc.weight.nelement())
    )
)
print(
    "Sparsity in fc.bias: {:.2f}%".format(
        100. * float(torch.sum(model.fc.bias == 0))
        / float(model.fc.bias.nelement())
    )
)
print(
    "Global sparsity: {:.2f}%".format(
        100. * float(
            torch.sum(model.embedding.weight == 0)
            + torch.sum(model.lstm.weight_ih_l0 == 0)
            + torch.sum(model.lstm.weight_hh_l0 == 0)
            + torch.sum(model.fc.weight == 0)
            + torch.sum(model.fc.bias == 0)
        )
        / float(
            model.embedding.weight.nelement()
            + model.lstm.weight_ih_l0.nelement()
            + model.lstm.weight_hh_l0.nelement()
            + model.fc.weight.nelement()
            + model.fc.bias.nelement()
        )
    )
)

Sparsity in embedding.weight: 68.92%
Sparsity in lstm.weight_ih_l0: 100.00%
Sparsity in lstm.weight_hh_l0: 100.00%
Sparsity in fc.weight: 100.00%
Sparsity in fc.bias: 100.00%
Global sparsity: 70.00%


#### Custom pruning function
To implement our own pruning function, we can extend the `nn.utils.prune` module by subclassing the `BasePruningMethod` base class, the same way all other pruning methods do. The base class implements the following methods for you: `__call__`, `apply_mask`, `apply`, `prune`, and `remove`. Beyond some special cases, you shouldn't have to reimplement these methods for your new pruning technique.

We will, however, have to implement `__init__` (the constructor), and `compute_mask` (the instructions on how to compute the mask for the given tensor according to the logic of your pruning technique).

In addition, we will have to specify which type of pruning this technique implements (supported options are `global, structured, and unstructured`). This is needed to determine how to combine masks in the case in which pruning is applied iteratively. In other words, when pruning a pre-pruned parameter, the current prunining techique is expected to act on the unpruned portion of the parameter. Specifying the `PRUNING_TYPE` will enable the `PruningContainer` (which handles the iterative application of pruning masks) to correctly identify the slice of the parameter to prune.

Let's assume, for example, that we want to implement a pruning technique that prunes every other entry in a tensor (or -- if the tensor has previously been pruned -- in the remaining unpruned portion of the tensor). This will be of PRUNING_TYPE='unstructured' because it acts on individual connections in a layer and not on entire units/channels ('structured'), or across different parameters ('global')

In [89]:
class CustomPruning(prune.BasePruningMethod):
    """Prune every other entry in a tensor
    """
    PRUNING_TYPE = 'unstructured'

    def compute_mask(self, t, default_mask):
        mask = default_mask.clone()
        mask.view(-1)[::2] = 0  # Flattening and masking
        return mask

`compute_mask` compute and returns a mask for the input tensor t.

Starting from a base default_mask (which should be a mask of ones if the tensor has not been pruned yet), generate a random mask to apply on top of the default_mask according to the specific pruning method recipe.

Now, to apply this to a parameter in an nn.Module, you should also provide a simple function that instantiates the method and applies it.

In [90]:
def prune_custom_unstructured(module, name):
    """Prunes tensor corresponding to parameter called `name` in `module`
    by removing every other entry in the tensors.
    Modifies module in place (and also return the modified module)
    by:
    1) adding a named buffer called `name+'_mask'` corresponding to the
    binary mask applied to the parameter `name` by the pruning method.
    The parameter `name` is replaced by its pruned version, while the
    original (unpruned) parameter is stored in a new parameter named
    `name+'_orig'`.

    Args:
        module (nn.Module): module containing the tensor to prune
        name (string): parameter name within `module` on which pruning
                will act.

    Returns:
        module (nn.Module): modified (i.e. pruned) version of the input
            module

    Examples:
        >>> m = nn.Linear(3, 4)
        >>> foobar_unstructured(m, name='bias')
    """
    CustomPruning.apply(module, name)
    return module

In [91]:
model = reset()
prune_custom_unstructured(model.fc, name='bias')

tensor([0., 1., 0., 1.])


In [92]:
print(model.fc)

Linear(in_features=512, out_features=4, bias=True)


In [99]:
print(model.fc.bias) # bias is now a attribute here, not parameter anymore.

tensor([-0.0000,  0.0169,  0.0000, -0.0252], grad_fn=<MulBackward0>)


In [98]:
print(model.fc.bias_mask)

tensor([0., 1., 0., 1.])


In [95]:
print(list(model.fc.named_parameters()))

[('weight', Parameter containing:
tensor([[ 0.0426,  0.0199, -0.0126,  ..., -0.0379, -0.0259, -0.0220],
        [-0.0389, -0.0013, -0.0246,  ...,  0.0429, -0.0094,  0.0007],
        [-0.0148,  0.0003,  0.0002,  ..., -0.0161, -0.0162,  0.0190],
        [ 0.0106,  0.0279,  0.0244,  ...,  0.0332, -0.0118,  0.0288]],
       requires_grad=True)), ('bias_orig', Parameter containing:
tensor([-0.0080,  0.0169,  0.0113, -0.0252], requires_grad=True))]


In [96]:
print(list(model.fc.named_buffers()))

[('bias_mask', tensor([0., 1., 0., 1.]))]


In practice, we may want to perform iterative pruning where we iteratively prune the lowest magnitude, perhaps after each finetuning epoch, until sparsity is reached. You will be very amazed how much can be pruned without much impact on the performance.